# 02 · Cosmology and skewers

Along a photon path the scale factor obeys da/dx = (H0/c) √(Ω_r + Ω_m a + Ω_k a² + Ω_Λ a⁴) / √(1 − k x²)
(notes `sec_cosmology.tex`, Eq. dadx_general). `Cosmology` holds a batch of backgrounds and integrates it with
`diffrax`; `SkewerBundle` places the sampling points of N_sk skewers, sets the photon directions and solves a(x)
along each one. Runtime: ~15 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, LABELS, FIGSIZES,
                                      add_panel_letters, plot_pipeline, plot_spectra, zoom_inset)
use_style()
%matplotlib inline

In [ ]:
plot_pipeline(('lines', 'skewers'), note="filled: the steps of this notebook"); plt.show()

In [ ]:
from skewerjax.core import Cosmology, SkewerBundle, phys, path_geometry, plane_points, grid_skewer_points
h = phys.H0_over_c                                                  # H0 / c [1/Mpc]
cosmo = Cosmology(H0=jnp.full(4, phys.H0), Omega_r_0=jnp.array([1.0, 0.0, 0.0, 9.1436e-5]), Omega_m_0=jnp.array([0.0, 1.0, 0.0, 0.3089]),
                  Omega_k_0=jnp.zeros(4), Omega_lambda_0=jnp.array([0.0, 0.0, 1.0, 0.6911]))
x_f, x_b = np.linspace(0, 1 / h - 1e-6, 600), np.linspace(0, -2 / h + 1e-6, 600)
a = np.concatenate([np.asarray(cosmo.solve_a_of_x(jnp.ones(4), jnp.asarray(x_b)))[:, ::-1], np.asarray(cosmo.solve_a_of_x(jnp.ones(4), jnp.asarray(x_f)))], 1)
x = np.concatenate([x_b[::-1], x_f])
exact = {"radiation": 1 + h * x, "matter": (1 + 0.5 * h * x) ** 2, "Lambda": 1 / (1 - h * x)}
fig, (ax, axr) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
for i, (name, c) in enumerate(zip(["radiation", "matter", "Lambda", "Planck 2018"], [TOL["red"], TOL["blue"], TOL["green"], "k"])):
    ax.plot(x, a[i], color=c, label=name)
    if name in exact:
        ok = np.isfinite(exact[name]) & (exact[name] > 0)
        ax.plot(x[ok], exact[name][ok], color=c, **MS["reference"])
        axr.plot(x[ok], a[i][ok] / exact[name][ok] - 1, color=c)
ax.plot(0, 1, "kx", ms=12, mew=3); ax.annotate("observer today", (0, 1), (300, 0.3), arrowprops=dict(arrowstyle="->"), fontsize=15)
ax.set(yscale="log", ylim=(1e-3, 1e2), ylabel="scale factor $a$"); ax.legend(loc="lower right")
axr.set(xlabel="comoving distance $x$ [Mpc] (towards the future)", ylabel="residual", yscale="symlog", ylim=(-1e-3, 1e-3))
axr.axhline(0, color="k", lw=1); add_panel_letters([ax, axr])
fig.suptitle("a(x) along a light ray: the ODE solution follows the closed forms (thick, light)", fontsize=22); plt.show()

The default tolerances (`rtol = atol = 1e-4`) give a(x) to ~10⁻⁴; `solve_a_of_x(..., rtol=1e-10, atol=1e-12)` to ~10⁻⁹ (`scripts/validation/check_cosmology.py`). The parameters are pytree leaves: `jax.grad` of any result with respect to Ω_m works.

## 1 · SkewerBundle: points, photon direction and a(x)

Each skewer has its initial condition (a0 at comoving distance x0) at its first (`ICs_defined_at_index=0`) or last
(`-1`) point, and that point is either where the light is emitted or where it is absorbed
(`ICs_at_emission_or_absorption`). These two choices fix the photon direction relative to the order of the points
(`array_direction`, ±1) and how a(x) is integrated.

In [ ]:
L = 1000.0
px = L * jnp.array([[0.0, 0.5, 0.5], [0.2, 0.5, 0.5], [0.3, 0.5, 0.5], [0.5, 0.5, 0.5], [1.0, 0.5, 0.5]])
planck = Cosmology(H0=phys.H0, Omega_r_0=9.1436e-5, Omega_m_0=0.3089, Omega_k_0=0.0, Omega_lambda_0=0.6911)
configs = [(0, "emission"), (0, "absorption"), (-1, "emission"), (-1, "absorption")]
sb = SkewerBundle(spatial_points=jnp.stack([px] * 4), a0=1.0, x0=0.0, ICs_defined_at_index=jnp.array([c[0] for c in configs]),
                  ICs_at_emission_or_absorption=[c[1] for c in configs], cosmologies=planck)
print("array_direction:", np.asarray(sb.array_direction), "| quadri_vectors (N_cosmo, N_sk, N_ppsk, 4):", sb.quadri_vectors.shape)
fig, ax = plt.subplots(figsize=(13, 6), layout="constrained")
for k, (i0, kind) in enumerate(configs):
    ax.plot(np.asarray(px[:, 0]), np.asarray(sb.scale_factors[0, k]), "o-", color=list(TOL.values())[k],
            label=f"ICs at point {i0} = {kind} (direction {int(sb.array_direction[k]):+d})")
ax.set(xlabel="x [Mpc]", ylabel="a(x)", title="The same points, four conventions: a = 1 where the ICs are set"); ax.legend(); plt.show()

## 2 · Geometry of a path

`path_geometry(points)` returns the path length s, the **line elements** w_i (half the distance to each neighbour, the
default quadrature weights of τ), the unit tangents and the midpoints m_{i±1/2} that bound the *dual segment* of every
point (where the simulation deposition of 05 puts the gas). Point layouts: `plane_points`, `grid_skewer_points`,
`axis_aligned_skewer_points`.

In [ ]:
zig = jnp.array([[0.0, 0.0, 0.0], [0.1, 0.2, 0.0], [0.35, 0.25, 0.0], [0.5, 0.6, 0.0], [0.9, 0.7, 0.0]])
g = path_geometry(zig)
fig, axs = plt.subplots(1, 2, figsize=(20, 7), layout="constrained")
ax = axs[0]
ax.plot(zig[:, 0], zig[:, 1], "o-", color="k", ms=10, label="points")
for i, (lo, p, hi) in enumerate(zip(np.asarray(g["mid_lo"]), np.asarray(zig), np.asarray(g["mid_hi"]))):
    ax.plot([lo[0], p[0], hi[0]], [lo[1], p[1], hi[1]], color=(TOL["blue"], TOL["cyan"])[i % 2], lw=9, alpha=0.6, solid_capstyle="butt")
ax.quiver(zig[:, 0], zig[:, 1], g["tangents"][:, 0], g["tangents"][:, 1], color=TOL["red"], scale=8, width=0.006, label="tangents")
for p, w in zip(np.asarray(zig), np.asarray(g["line_elements"])):
    ax.annotate(f"w = {w:.3f}", p[:2], (8, -18), textcoords="offset points", fontsize=13)
ax.set(xlabel="x [cMpc]", ylabel="y [cMpc]", aspect="equal", title="dual segments (alternating colours) tile the path")
grid = grid_skewer_points(10.0, 4, 32, direction="z")
axs[1].scatter(grid[..., 0].ravel(), grid[..., 1].ravel(), c=TOL["blue"], s=60)
axs[1].set(xlabel="x [cMpc]", ylabel="y [cMpc]", aspect="equal", title="grid_skewer_points: 4 × 4 skewers along z (cell centres)")
add_panel_letters(axs); plt.show()
print(f"sum of line elements {float(g['line_elements'].sum()):.6f} = path length {float(g['s'][-1]):.6f}")

**Options** of `SkewerBundle`: `a0`, `x0`, `ICs_defined_at_index` and `ICs_at_emission_or_absorption` (scalars or
per skewer), `los_directions` (else from the points), `manual_scale_factors` (else solved with `cosmologies`, which
may hold several backgrounds: one set of quadri-vectors per cosmology). The bundle is a pytree.

## Summary

- a(x) along each skewer is solved with diffrax and matches the closed forms to the ODE tolerance.
- The position of the initial conditions and the emission/absorption convention fix the photon direction
  (`array_direction`).
- Line elements tile the path exactly. They are the quadrature weights of τ and the dual segments of the deposition.
- Checks: `scripts/validation/check_cosmology.py`.